# 03 — Corrected Baseline Research with Full MISO MCP Data

**Date:** 2026-02-11  
**Goal:** Rerun ALL baseline and residual research using the correct MCP pipeline.

## Why This Notebook Exists

Our previous analysis (notebook 02) had a **critical flaw in how we obtained R2/R3 baselines**:

- **Old (wrong) approach:** We manually matched R2 trades to R1 MCPs by joining on `path` within our own cleared trades. This only finds MCPs for paths that *we* traded in the previous round.
  - Result: **68% R2 coverage, 75% R3 coverage** — the 'missing' 32%/25% were paths where we didn't trade in the prior round.

- **New (correct) approach:** Use `aptools.tools.get_m2m_mcp_for_trades_all(trades)`, which looks up MCPs from MISO's **full published market data** — not just our trades. MISO publishes MCPs for ALL paths in every round.
  - Expected result: **~100% R2/R3 coverage**.

Additionally, there is a **dtype bug**: `get_all_cleared_trades()` returns some columns as pandas `category` dtype, which causes `"Object with dtype category cannot perform the numpy op multiply"` when doing math. We fix this with `aptools.tools.cast_category_to_str(trades)` immediately after loading.

---

## Pipeline Overview

```
Step 1: Load all annual cleared trades PY 2019-2025
        -> Fix category dtype bug
        -> Save annual_cleared_all_v2.parquet

Step 2: Get correct MTM/MCP for ALL trades via get_m2m_mcp_for_trades_all()
        -> This adds mtm_1st_mean, mtm_2nd_mean, mtm_3rd_mean, mcp_mean columns
        -> R2 gets R1's MCP, R3 gets R2's MCP (from full MISO data)
        -> R1 gets NaN for mtm_1st_mean (no prior round) — expected
        -> Save annual_with_mcp_v2.parquet

Step 3: Fill R1 baseline using historical DA congestion (H)
        -> fill_mtm_1st_period_with_hist_revenue()
        -> Save r1_filled_v2.parquet

Step 4: Compute residuals (all rounds)
        -> residual = mcp_mean - mtm_1st_mean
        -> Save all_residuals_v2.parquet

Step 5: Compute all statistics tables
        -> Coverage report
        -> By Round x Quarter
        -> By Round x Class Type
        -> R1 by |H| bin
        -> R2/R3 by |M| bin
        -> f0p comparison

Step 6: R2/R3 blend test (M vs M+H)
```

---
## Step 1: Load Annual Cleared Trades (with dtype fix)

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
os.chdir('/home/xyz/workspace/pmodel/src')

from pbase.config.ray import init_ray
import pmodel
init_ray(extra_modules=[pmodel])

from pbase.analysis.utils import misc
misc.logging_setup()

import pandas as pd
import numpy as np

pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_columns', 99)
pd.set_option('display.max_rows', 200)

from pbase.analysis.tools.all_positions import MisoApTools
aptools = MisoApTools()
print('Initialized')

In [ ]:
# Load all annual cleared trades PY 2019-2025
# Each planning year (PY) runs from June 1 to May 31 of the next year.
# We load 7 years of data to get a robust statistical picture.

trades_list = []
for py in range(2019, 2026):
    print(f'Loading PY {py}...')
    t = aptools.get_all_cleared_trades(
        start_date=pd.Timestamp(f"{py}-06-01"),
        end_date=pd.Timestamp(f"{py+1}-06-01"),
    )
    print(f'  -> {len(t)} rows')
    trades_list.append(t)

trades_all = pd.concat(trades_list, ignore_index=True)
print(f'\nTotal rows loaded: {len(trades_all)}')
print(f'Period types: {trades_all["period_type"].value_counts().to_dict()}')

# Filter to annual quarters only (aq1-aq4)
annual = trades_all[trades_all["period_type"].isin(["aq1","aq2","aq3","aq4"])].copy()
print(f'Annual rows (aq1-aq4 only): {len(annual)}')
print(f'Rounds: {annual["round"].value_counts().sort_index().to_dict()}')

In [ ]:
# FIX: Convert categorical columns to proper dtypes
# get_all_cleared_trades() returns some columns as pandas `category` dtype,
# which causes "Object with dtype category cannot perform the numpy op multiply"
# when doing math. cast_category_to_str converts category -> str and round -> int.

cat_cols_before = annual.select_dtypes('category').columns.tolist()
print(f'Category columns before fix: {cat_cols_before}')

annual = aptools.tools.cast_category_to_str(annual)

cat_cols_after = annual.select_dtypes('category').columns.tolist()
print(f'Category columns after fix: {cat_cols_after}')
print(f'round dtype: {annual["round"].dtype}')
print(f'\nShape: {annual.shape}')
print(f'Columns: {annual.columns.tolist()}')

In [ ]:
# Save cleaned annual trades
outpath = '/opt/temp/qianli/annual_research/annual_cleared_all_v2.parquet'
annual.to_parquet(outpath)
print(f'Saved {len(annual)} rows to {outpath}')
print(f'File size: {os.path.getsize(outpath) / 1e6:.1f} MB')

---
## Step 2: Get Correct MTM/MCP for ALL Trades

### What `get_m2m_mcp_for_trades_all()` does

This function looks up MCPs from MISO's **full published market clearing data**, not just our portfolio's trades:

- For **R2 trades**: it finds the R1 MCP for the same path from MISO's published results
- For **R3 trades**: it finds the R2 MCP for the same path
- For **R1 trades**: returns NaN (no prior round exists) — expected behavior

The key difference from our previous approach:
- **Old:** matched on `path` within our own cleared trades (only finds MCPs for paths we traded in both rounds)
- **New:** looks up from MISO's full market data (finds MCPs for ALL paths, since MISO publishes clearing prices for every path)

**Columns added:**
- `mtm_1st_mean`: MCP from the immediately prior round (R2 gets R1's MCP, R3 gets R2's MCP)
- `mtm_2nd_mean`: MCP from two rounds prior (R3 gets R1's MCP)
- `mtm_3rd_mean`: MCP from three rounds prior
- `mcp_mean`: the actual clearing price for this trade's round

In [ ]:
# Get correct MTM/MCP for ALL trades
# n_jobs=6 for parallel processing across planning years
# This is the CRITICAL fix: uses full MISO market data instead of just our trades

print('Running get_m2m_mcp_for_trades_all (this may take a few minutes)...')
annual_with_mcp = aptools.tools.get_m2m_mcp_for_trades_all(
    trades_all=annual,
    n_jobs=6
)
print(f'Done. Shape: {annual_with_mcp.shape}')
print(f'\nNew columns added: {[c for c in annual_with_mcp.columns if c not in annual.columns]}')

In [ ]:
# *** COVERAGE REPORT ***
# This is the key validation: R2/R3 should now have ~100% mtm_1st_mean coverage
# (Previously we had 68% R2, 75% R3 with the manual matching approach)

print('=== MTM Coverage Report (mtm_1st_mean) ===')
print('='*60)
for rnd in sorted(annual_with_mcp['round'].unique()):
    rnd_data = annual_with_mcp[annual_with_mcp['round'] == rnd]
    total = len(rnd_data)
    has_mtm = rnd_data['mtm_1st_mean'].notna().sum()
    pct = 100 * has_mtm / total if total > 0 else 0
    print(f'  Round {rnd}: {has_mtm:>10,} / {total:>10,} = {pct:.1f}%')

print(f'\nTotal rows: {len(annual_with_mcp):,}')
print(f'Total with mtm_1st_mean: {annual_with_mcp["mtm_1st_mean"].notna().sum():,}')

# Also check mcp_mean coverage
print(f'\n=== MCP Coverage Report (mcp_mean) ===')
for rnd in sorted(annual_with_mcp['round'].unique()):
    rnd_data = annual_with_mcp[annual_with_mcp['round'] == rnd]
    total = len(rnd_data)
    has_mcp = rnd_data['mcp_mean'].notna().sum()
    pct = 100 * has_mcp / total if total > 0 else 0
    print(f'  Round {rnd}: {has_mcp:>10,} / {total:>10,} = {pct:.1f}%')

In [ ]:
# Save annual trades with correct MCP data
outpath = '/opt/temp/qianli/annual_research/annual_with_mcp_v2.parquet'
annual_with_mcp.to_parquet(outpath)
print(f'Saved {len(annual_with_mcp)} rows to {outpath}')
print(f'File size: {os.path.getsize(outpath) / 1e6:.1f} MB')

---
## Step 3: Fill R1 Baseline with Historical DA Congestion (H)

### How `fill_mtm_1st_period_with_hist_revenue()` works

For R1 trades, `get_m2m_mcp_for_trades_all` returns NaN for `mtm_1st_mean` because there is no prior round. We fill this using the only available information: **historical day-ahead (DA) congestion prices**.

The function computes H (the historical baseline) by:
1. Taking each path's source and sink nodes
2. Loading monthly-average DA congestion prices for the relevant delivery months
   - e.g., for aq1 (Jun-Aug delivery), it loads Jun/Jul/Aug DA congestion from the prior year
3. Only using months before the April cutoff date (this is why aq4 gets less data)
4. Applying a 0.85 shrinkage factor to congestion prices in the profitable direction
5. Setting `mtm_1st_mean = sink_congestion - source_congestion`

This function must be called per group of `(auction_date, period_type, class_type)`.

In [ ]:
# Extract R1 trades (mtm_1st_mean is NaN — expected)
r1 = annual_with_mcp[annual_with_mcp['round'] == 1].copy()
print(f'R1 trades: {len(r1)}')
print(f'R1 mtm_1st_mean NaN: {r1["mtm_1st_mean"].isna().sum()} ({100*r1["mtm_1st_mean"].isna().mean():.1f}%)')

# Fill H baseline for R1 trades
# Group by (auction_date, period_type, class_type) as required by fill_mtm
r1_filled_list = []
groups = list(r1.groupby(['auction_date', 'period_type', 'class_type']))
print(f'Number of groups: {len(groups)}')

for i, (keys, group) in enumerate(groups):
    if i % 10 == 0:
        print(f'  Processing group {i+1}/{len(groups)}: {keys} ({len(group)} rows)...')
    try:
        filled = aptools.tools.fill_mtm_1st_period_with_hist_revenue(group)
        r1_filled_list.append(filled)
    except Exception as e:
        print(f'  ERROR on {keys}: {e}')

r1_filled = pd.concat(r1_filled_list, ignore_index=True)
print(f'\nR1 filled total: {len(r1_filled)}')
print(f'R1 with H (mtm_1st_mean not NaN): {r1_filled["mtm_1st_mean"].notna().sum()}')
print(f'R1 coverage: {100*r1_filled["mtm_1st_mean"].notna().mean():.1f}%')

In [ ]:
# Save R1 with filled H baseline
outpath = '/opt/temp/qianli/annual_research/r1_filled_v2.parquet'
r1_filled.to_parquet(outpath)
print(f'Saved {len(r1_filled)} rows to {outpath}')
print(f'File size: {os.path.getsize(outpath) / 1e6:.1f} MB')

---
## Step 4: Compute Residuals (All Rounds)

### Residual Definition

```
residual = mcp_mean - mtm_1st_mean
```

Where:
- `mcp_mean` = actual clearing price for this trade's round
- `mtm_1st_mean` = baseline prediction:
  - **R1:** H (historical DA congestion, filled in Step 3)
  - **R2:** R1's MCP (from Step 2, full MISO market data)
  - **R3:** R2's MCP (from Step 2, full MISO market data)

The residual measures how far the actual clearing price is from the baseline. Smaller residuals = better baseline prediction = narrower bid bands needed.

In [ ]:
# Combine R1 (with H baseline from Step 3) and R2/R3 (with M baseline from Step 2)
r23 = annual_with_mcp[annual_with_mcp['round'].isin([2, 3])].copy()
r23_before = len(r23)
r23 = r23.dropna(subset=['mtm_1st_mean'])  # drop any truly missing
print(f'R2/R3 before dropna: {r23_before}')
print(f'R2/R3 after dropna: {len(r23)} (dropped {r23_before - len(r23)})')

# Combine
all_with_baseline = pd.concat([r1_filled, r23], ignore_index=True)
print(f'\nCombined rows: {len(all_with_baseline)}')

# Compute residuals
all_with_baseline['residual'] = all_with_baseline['mcp_mean'] - all_with_baseline['mtm_1st_mean']
all_with_baseline['abs_residual'] = all_with_baseline['residual'].abs()

# Drop rows where residual couldn't be computed (R1 rows missing H)
valid_mask = all_with_baseline['residual'].notna()
print(f'Rows with valid residual: {valid_mask.sum()} ({100*valid_mask.mean():.1f}%)')
all_with_baseline = all_with_baseline[valid_mask].copy()

# Summary
print(f'\n=== Residual Summary ===')
for rnd in sorted(all_with_baseline['round'].unique()):
    rnd_data = all_with_baseline[all_with_baseline['round'] == rnd]
    print(f'R{rnd}: {len(rnd_data):>10,} rows, '
          f'mean residual = {rnd_data["residual"].mean():+.0f}, '
          f'mean |residual| = {rnd_data["abs_residual"].mean():.0f}, '
          f'p95 |residual| = {rnd_data["abs_residual"].quantile(0.95):.0f}')

In [ ]:
# Save all residuals
outpath = '/opt/temp/qianli/annual_research/all_residuals_v2.parquet'
all_with_baseline.to_parquet(outpath)
print(f'Saved {len(all_with_baseline)} rows to {outpath}')
print(f'File size: {os.path.getsize(outpath) / 1e6:.1f} MB')

---
## Step 5: Compute All Residual Statistics

We reproduce every table from the previous `findings_residuals.md` with corrected data.

In [ ]:
# Helper function for residual statistics
def residual_stats(df, groupby_cols):
    """Compute standard residual statistics grouped by specified columns.
    
    Returns: n, bias (mean residual), mean/median |residual|, p90/p95/p99, direction accuracy.
    """
    # Direction accuracy: does sign(baseline) == sign(mcp)?
    df = df.copy()
    df['sign_match'] = (np.sign(df['mcp_mean']) == np.sign(df['mtm_1st_mean'])).astype(int)
    
    result = df.groupby(groupby_cols).agg(
        n=('residual', 'size'),
        bias=('residual', 'mean'),
        mean_abs_res=('abs_residual', 'mean'),
        median_abs_res=('abs_residual', 'median'),
        p90_abs_res=('abs_residual', lambda x: x.quantile(0.90)),
        p95_abs_res=('abs_residual', lambda x: x.quantile(0.95)),
        p99_abs_res=('abs_residual', lambda x: x.quantile(0.99)),
        dir_accuracy=('sign_match', 'mean'),
    ).reset_index()
    
    # Format
    for col in ['bias', 'mean_abs_res', 'median_abs_res', 'p90_abs_res', 'p95_abs_res', 'p99_abs_res']:
        result[col] = result[col].round(0).astype(int)
    result['dir_accuracy'] = (result['dir_accuracy'] * 100).round(0).astype(int)
    
    return result

print('Helper function defined.')

In [ ]:
# 5.1 Coverage Report
# Compare old vs new coverage
print('='*70)
print('5.1 COVERAGE REPORT: Old (manual match) vs New (full MISO MCP data)')
print('='*70)

print(f'\n{"Round":<8} {"Old Coverage":>15} {"New Coverage":>15} {"Improvement":>15}')
print('-' * 55)

old_coverage = {1: 100.0, 2: 68.0, 3: 75.1}  # from previous analysis
for rnd in [1, 2, 3]:
    rnd_data = annual_with_mcp[annual_with_mcp['round'] == rnd]
    total = len(rnd_data)
    if rnd == 1:
        # R1 coverage is based on r1_filled
        has = r1_filled['mtm_1st_mean'].notna().sum()
        total_r1 = len(r1_filled)
        new_pct = 100 * has / total_r1
    else:
        has = rnd_data['mtm_1st_mean'].notna().sum()
        new_pct = 100 * has / total
    old_pct = old_coverage[rnd]
    improvement = new_pct - old_pct
    print(f'R{rnd:<7} {old_pct:>14.1f}% {new_pct:>14.1f}% {improvement:>+14.1f}%')

In [ ]:
# 5.2 By Round x Quarter
print('='*70)
print('5.2 RESIDUAL STATS BY ROUND x QUARTER (all PYs, both class types)')
print('='*70)

all_with_baseline['round_label'] = 'R' + all_with_baseline['round'].astype(str)
stats_rq = residual_stats(all_with_baseline, ['round_label', 'period_type'])
stats_rq = stats_rq.sort_values(['round_label', 'period_type'])

print(f'\n{"Round":<6} {"Quarter":<8} {"n":>10} {"Bias":>8} {"Mean|R|":>8} {"Med|R|":>8} {"p90|R|":>8} {"p95|R|":>8} {"p99|R|":>8} {"DirAcc":>7}')
print('-' * 85)
for _, row in stats_rq.iterrows():
    print(f'{row["round_label"]:<6} {row["period_type"]:<8} {row["n"]:>10,} {row["bias"]:>+8} {row["mean_abs_res"]:>8} {row["median_abs_res"]:>8} {row["p90_abs_res"]:>8} {row["p95_abs_res"]:>8} {row["p99_abs_res"]:>8} {row["dir_accuracy"]:>6}%')

In [ ]:
# 5.3 By Round x Class Type
print('='*70)
print('5.3 RESIDUAL STATS BY ROUND x CLASS TYPE')
print('='*70)

stats_rc = residual_stats(all_with_baseline, ['round_label', 'class_type'])
stats_rc = stats_rc.sort_values(['round_label', 'class_type'])

print(f'\n{"Round":<6} {"Class":<10} {"n":>10} {"Bias":>8} {"Mean|R|":>8} {"Med|R|":>8} {"p90|R|":>8} {"p95|R|":>8} {"p99|R|":>8}')
print('-' * 80)
for _, row in stats_rc.iterrows():
    print(f'{row["round_label"]:<6} {row["class_type"]:<10} {row["n"]:>10,} {row["bias"]:>+8} {row["mean_abs_res"]:>8} {row["median_abs_res"]:>8} {row["p90_abs_res"]:>8} {row["p95_abs_res"]:>8} {row["p99_abs_res"]:>8}')

In [ ]:
# 5.4 R1 by |H| bin
print('='*70)
print('5.4 R1 RESIDUAL STATS BY |H| BIN')
print('='*70)

r1_data = all_with_baseline[all_with_baseline['round'] == 1].copy()
r1_data['abs_H'] = r1_data['mtm_1st_mean'].abs()

bins = [0, 50, 250, 1000, float('inf')]
labels = ['tiny(<50)', 'small(50-250)', 'medium(250-1k)', 'large(1k+)']
r1_data['h_bin'] = pd.cut(r1_data['abs_H'], bins=bins, labels=labels, right=False)

stats_r1_bin = residual_stats(r1_data, ['h_bin'])

print(f'\n{"Bin":<18} {"n":>10} {"Bias":>8} {"Mean|R|":>8} {"Med|R|":>8} {"p90|R|":>8} {"p95|R|":>8} {"p99|R|":>8} {"DirAcc":>7}')
print('-' * 90)
for _, row in stats_r1_bin.iterrows():
    print(f'{str(row["h_bin"]):<18} {row["n"]:>10,} {row["bias"]:>+8} {row["mean_abs_res"]:>8} {row["median_abs_res"]:>8} {row["p90_abs_res"]:>8} {row["p95_abs_res"]:>8} {row["p99_abs_res"]:>8} {row["dir_accuracy"]:>6}%')

In [ ]:
# 5.5 R2/R3 by |M| bin (for f2p feasibility)
print('='*70)
print('5.5 R2/R3 RESIDUAL STATS BY |M| BIN')
print('='*70)

r23_data = all_with_baseline[all_with_baseline['round'].isin([2, 3])].copy()
r23_data['abs_M'] = r23_data['mtm_1st_mean'].abs()

bins = [0, 50, 250, 1000, float('inf')]
labels = ['tiny(<50)', 'small(50-250)', 'medium(250-1k)', 'large(1k+)']
r23_data['m_bin'] = pd.cut(r23_data['abs_M'], bins=bins, labels=labels, right=False)

stats_r23_bin = residual_stats(r23_data, ['round_label', 'class_type', 'm_bin'])

print(f'\n{"Round":<6} {"Class":<10} {"Bin":<18} {"n":>10} {"Bias":>8} {"Mean|R|":>8} {"p90|R|":>8} {"p95|R|":>8} {"DirAcc":>7}')
print('-' * 90)
for _, row in stats_r23_bin.iterrows():
    print(f'{row["round_label"]:<6} {row["class_type"]:<10} {str(row["m_bin"]):<18} {row["n"]:>10,} {row["bias"]:>+8} {row["mean_abs_res"]:>8} {row["p90_abs_res"]:>8} {row["p95_abs_res"]:>8} {row["dir_accuracy"]:>6}%')

# Also show avg per PY for feasibility
print(f'\n--- Avg rows per PY per bin ---')
n_pys = r23_data['planning_year'].nunique() if 'planning_year' in r23_data.columns else 7
for _, row in stats_r23_bin.iterrows():
    avg_per_py = row['n'] / n_pys
    print(f'{row["round_label"]} {row["class_type"]:10} {str(row["m_bin"]):18} avg/PY = {avg_per_py:>8,.0f}')

In [ ]:
# 5.6 f0p Residual Comparison
# Load f0p data for comparison
print('='*70)
print('5.6 f0p RESIDUAL COMPARISON')
print('='*70)

import glob as glob_module

f0p_files = glob_module.glob('/opt/temp/qianli/mcp_pred_training/auction_month=*/period_type=f0/class_type=*/training.parquet')
print(f'Found {len(f0p_files)} f0p training files')

if len(f0p_files) > 0:
    f0p_dfs = []
    for f in f0p_files:
        df = pd.read_parquet(f)
        f0p_dfs.append(df)
    f0p_data = pd.concat(f0p_dfs, ignore_index=True)
    f0p_data['residual'] = f0p_data['mcp_mean'] - f0p_data['mtm_1st_mean']
    f0p_data['abs_residual'] = f0p_data['residual'].abs()
    
    f0p_stats = f0p_data.groupby('class_type').agg(
        n=('residual', 'size'),
        bias=('residual', lambda x: int(round(x.mean()))),
        mean_abs=('abs_residual', lambda x: int(round(x.mean()))),
        median_abs=('abs_residual', lambda x: int(round(x.median()))),
        p90_abs=('abs_residual', lambda x: int(round(x.quantile(0.90)))),
        p95_abs=('abs_residual', lambda x: int(round(x.quantile(0.95)))),
        p99_abs=('abs_residual', lambda x: int(round(x.quantile(0.99)))),
    ).reset_index()
    
    print(f'\nf0p Residual Stats:')
    print(f0p_stats.to_string(index=False))
    
    # Side-by-side comparison
    print(f'\n--- Direct Comparison: Annual vs f0p |Residual| ---')
    
    # Compute overall annual stats per round
    annual_by_round = all_with_baseline.groupby('round').agg(
        mean_abs=('abs_residual', lambda x: int(round(x.mean()))),
        median_abs=('abs_residual', lambda x: int(round(x.median()))),
        p90_abs=('abs_residual', lambda x: int(round(x.quantile(0.90)))),
        p95_abs=('abs_residual', lambda x: int(round(x.quantile(0.95)))),
        p99_abs=('abs_residual', lambda x: int(round(x.quantile(0.99)))),
    ).reset_index()
    
    f0p_overall = f0p_data.agg(
        mean_abs=('abs_residual', lambda x: int(round(x.mean()))),
        median_abs=('abs_residual', lambda x: int(round(x.median()))),
        p90_abs=('abs_residual', lambda x: int(round(x.quantile(0.90)))),
        p95_abs=('abs_residual', lambda x: int(round(x.quantile(0.95)))),
        p99_abs=('abs_residual', lambda x: int(round(x.quantile(0.99)))),
    )
    
    # Print comparison
    print(f'\n{"Metric":<15} {"R1 (H)":>10} {"R2 (M)":>10} {"R3 (M)":>10} {"f0p (MTM)":>10}')
    print('-' * 58)
    for metric in ['mean_abs', 'median_abs', 'p90_abs', 'p95_abs', 'p99_abs']:
        r1_val = annual_by_round.loc[annual_by_round['round']==1, metric].values[0]
        r2_val = annual_by_round.loc[annual_by_round['round']==2, metric].values[0]
        r3_val = annual_by_round.loc[annual_by_round['round']==3, metric].values[0]
        f0p_val = int(round(f0p_data[f'abs_residual'].agg(metric.replace('_abs', '') if 'mean' in metric else (lambda x: x.quantile(float('0.' + metric.split('_')[0][1:]))))))
        print(f'{metric:<15} {r1_val:>10,} {r2_val:>10,} {r3_val:>10,}')
else:
    # Fallback: load f0p from our parquet
    print('No f0p training files found. Loading from cached f0p parquet...')
    f0p_path = '/opt/temp/qianli/annual_research/f0p_cleared_all.parquet'
    if os.path.exists(f0p_path):
        f0p_data = pd.read_parquet(f0p_path)
        print(f'Loaded {len(f0p_data)} f0p rows')
        # f0p data from cleared trades has mcp_mean and mtm_1st_mean columns
        if 'mcp_mean' in f0p_data.columns and 'mtm_1st_mean' in f0p_data.columns:
            f0p_data['residual'] = f0p_data['mcp_mean'] - f0p_data['mtm_1st_mean']
            f0p_data['abs_residual'] = f0p_data['residual'].abs()
            f0p_valid = f0p_data.dropna(subset=['residual'])
            print(f'f0p with valid residual: {len(f0p_valid)}')
    else:
        print(f'No f0p data available at {f0p_path}')

In [ ]:
# Better f0p comparison using the f0p cleared trades parquet
# The f0p_cleared_all.parquet should have mtm_1st_mean from the f0p pipeline
print('='*70)
print('5.6b f0p COMPARISON (from f0p cleared trades)')
print('='*70)

f0p_path = '/opt/temp/qianli/annual_research/f0p_cleared_all.parquet'
if os.path.exists(f0p_path):
    f0p_data = pd.read_parquet(f0p_path)
    print(f'Loaded f0p rows: {len(f0p_data)}')
    print(f'Columns: {[c for c in f0p_data.columns if "mtm" in c or "mcp" in c]}')
    
    # Check if it has the right columns
    if 'mcp_mean' in f0p_data.columns and 'mtm_1st_mean' in f0p_data.columns:
        f0p_data['residual'] = f0p_data['mcp_mean'] - f0p_data['mtm_1st_mean']
        f0p_data['abs_residual'] = f0p_data['residual'].abs()
        f0p_valid = f0p_data.dropna(subset=['residual'])
        
        f0p_overall = pd.DataFrame([{
            'n': len(f0p_valid),
            'bias': int(round(f0p_valid['residual'].mean())),
            'mean_abs': int(round(f0p_valid['abs_residual'].mean())),
            'median_abs': int(round(f0p_valid['abs_residual'].median())),
            'p90_abs': int(round(f0p_valid['abs_residual'].quantile(0.90))),
            'p95_abs': int(round(f0p_valid['abs_residual'].quantile(0.95))),
            'p99_abs': int(round(f0p_valid['abs_residual'].quantile(0.99))),
        }])
        print(f'\nf0p Overall Residual Stats:')
        print(f0p_overall.to_string(index=False))
        
        # By class_type
        if 'class_type' in f0p_data.columns:
            f0p_by_class = f0p_valid.groupby('class_type').agg(
                n=('residual', 'size'),
                bias=('residual', lambda x: int(round(x.mean()))),
                mean_abs=('abs_residual', lambda x: int(round(x.mean()))),
                median_abs=('abs_residual', lambda x: int(round(x.median()))),
                p90_abs=('abs_residual', lambda x: int(round(x.quantile(0.90)))),
                p95_abs=('abs_residual', lambda x: int(round(x.quantile(0.95)))),
                p99_abs=('abs_residual', lambda x: int(round(x.quantile(0.99)))),
            ).reset_index()
            print(f'\nf0p by class_type:')
            print(f0p_by_class.to_string(index=False))
    elif 'mcp' in f0p_data.columns:
        print('f0p data has mcp but not mcp_mean. Using mcp column...')
        # The f0p cleared trades might use different column names
        print(f'Available MCP/MTM columns: {[c for c in f0p_data.columns if "mcp" in c.lower() or "mtm" in c.lower()]}')
else:
    print(f'File not found: {f0p_path}')

---
## Step 6: R2/R3 Blend Test

### Motivation

For R2/R3, we have two potential baselines:
- **M** = previous round's MCP (from full MISO market data)
- **H** = historical DA congestion (same as R1 baseline)

Should we use M alone, or blend M with H? The hypothesis is that M alone is optimal because the previous round's MCP already incorporates all the information that H provides (and more). Adding H would just add noise.

We test three blend ratios:
- M=1.0, H=0.0 (M only — current approach)
- M=0.95, H=0.05 (5% H blend)
- M=0.90, H=0.10 (10% H blend)

In [ ]:
# R2/R3 Blend Test: M-only vs M+H blends
print('='*70)
print('STEP 6: R2/R3 BLEND TEST')
print('='*70)

# We need H values for R2/R3 trades. Get them from the r1_filled data.
# First, build a lookup of H values by path
# H is computed per (path, period_type, class_type, planning_year)
r1_h_lookup = r1_filled.groupby(['planning_year', 'period_type', 'class_type', 'path'])['mtm_1st_mean'].mean().reset_index()
r1_h_lookup = r1_h_lookup.rename(columns={'mtm_1st_mean': 'H_baseline'})

# Merge H onto R2/R3 data
r23_blend = r23_data.copy()
if 'path' in r23_blend.columns:
    merge_cols = ['planning_year', 'period_type', 'class_type', 'path']
else:
    # Try source_id/sink_id
    r1_h_lookup2 = r1_filled.groupby(['planning_year', 'period_type', 'class_type', 'source_id', 'sink_id'])['mtm_1st_mean'].mean().reset_index()
    r1_h_lookup2 = r1_h_lookup2.rename(columns={'mtm_1st_mean': 'H_baseline'})
    merge_cols = ['planning_year', 'period_type', 'class_type', 'source_id', 'sink_id']
    r1_h_lookup = r1_h_lookup2

r23_blend = r23_blend.merge(r1_h_lookup, on=merge_cols, how='left')
r23_blend_valid = r23_blend.dropna(subset=['H_baseline', 'mtm_1st_mean', 'mcp_mean'])
print(f'R2/R3 with both M and H: {len(r23_blend_valid)} ({100*len(r23_blend_valid)/len(r23_blend):.1f}%)')

# Test blends
blends = [(1.0, 0.0), (0.95, 0.05), (0.90, 0.10)]

print(f'\n{"Blend (M,H)":<15} {"Round":<6} {"n":>10} {"Mean|R|":>10} {"p90|R|":>10} {"p95|R|":>10} {"p99|R|":>10}')
print('-' * 70)

for m_wt, h_wt in blends:
    r23_blend_valid[f'blend_{m_wt}'] = m_wt * r23_blend_valid['mtm_1st_mean'] + h_wt * r23_blend_valid['H_baseline']
    r23_blend_valid[f'blend_res_{m_wt}'] = r23_blend_valid['mcp_mean'] - r23_blend_valid[f'blend_{m_wt}']
    r23_blend_valid[f'blend_abs_{m_wt}'] = r23_blend_valid[f'blend_res_{m_wt}'].abs()
    
    for rnd in [2, 3]:
        rnd_data = r23_blend_valid[r23_blend_valid['round'] == rnd]
        print(f'M={m_wt},H={h_wt}  R{rnd:<4} {len(rnd_data):>10,} '
              f'{rnd_data[f"blend_abs_{m_wt}"].mean():>10.0f} '
              f'{rnd_data[f"blend_abs_{m_wt}"].quantile(0.90):>10.0f} '
              f'{rnd_data[f"blend_abs_{m_wt}"].quantile(0.95):>10.0f} '
              f'{rnd_data[f"blend_abs_{m_wt}"].quantile(0.99):>10.0f}')

In [ ]:
# 5.7 Detailed per-Round x per-Quarter stats (for findings_legacy_pricing.md Addition B)
print('='*70)
print('5.7 DETAILED PER-ROUND x PER-QUARTER GRID')
print('='*70)

for rnd in [1, 2, 3]:
    for qt in ['aq1', 'aq2', 'aq3', 'aq4']:
        subset = all_with_baseline[(all_with_baseline['round'] == rnd) & (all_with_baseline['period_type'] == qt)]
        if len(subset) == 0:
            continue
        
        n = len(subset)
        if 'path' in subset.columns:
            n_paths = subset['path'].nunique()
        elif 'source_id' in subset.columns and 'sink_id' in subset.columns:
            n_paths = subset.groupby(['source_id', 'sink_id']).ngroups
        else:
            n_paths = 'N/A'
        
        bias = subset['residual'].mean()
        mean_abs = subset['abs_residual'].mean()
        median_abs = subset['abs_residual'].median()
        p90 = subset['abs_residual'].quantile(0.90)
        p95 = subset['abs_residual'].quantile(0.95)
        p99 = subset['abs_residual'].quantile(0.99)
        
        sign_match = (np.sign(subset['mcp_mean']) == np.sign(subset['mtm_1st_mean'])).mean()
        
        print(f'\n--- R{rnd} x {qt} ---')
        print(f'  Trades: {n:,} | Unique paths: {n_paths}')
        print(f'  Bias: {bias:+.0f} | Mean |res|: {mean_abs:.0f} | Median |res|: {median_abs:.0f}')
        print(f'  p90: {p90:.0f} | p95: {p95:.0f} | p99: {p99:.0f}')
        print(f'  Direction accuracy: {100*sign_match:.0f}%')
        
        # Compute legacy bid range for a typical path
        median_baseline = subset['mtm_1st_mean'].abs().median()
        # Legacy: min bid = 0.5*H - 50, max bid = H + min(1.5*H + 150, 2000)
        # Range = max - min
        legacy_min = 0.5 * median_baseline - 50
        legacy_max_adj = min(1.5 * median_baseline + 150, 2000)
        legacy_range = legacy_max_adj + 0.5 * median_baseline + 50
        print(f'  Median |baseline|: {median_baseline:.0f}')
        print(f'  Legacy bid range (for median |baseline|): {legacy_range:.0f}')
        print(f'  Range needed for p95 coverage: {2 * p95:.0f}')
        print(f'  Range adequacy: {100 * legacy_range / (2 * p95):.0f}% of p95 range')

In [ ]:
# R1 detailed: by quarter x class_type (for findings_residuals.md section 2.4)
print('='*70)
print('R1 RESIDUAL STATS BY QUARTER x CLASS TYPE')
print('='*70)

stats_r1_qc = residual_stats(r1_data, ['period_type', 'class_type'])
print(f'\n{"Quarter":<8} {"Class":<10} {"n":>10} {"Bias":>8} {"Mean|R|":>8} {"Med|R|":>8} {"p90|R|":>8} {"p95|R|":>8} {"DirAcc":>7}')
print('-' * 80)
for _, row in stats_r1_qc.iterrows():
    print(f'{row["period_type"]:<8} {row["class_type"]:<10} {row["n"]:>10,} {row["bias"]:>+8} {row["mean_abs_res"]:>8} {row["median_abs_res"]:>8} {row["p90_abs_res"]:>8} {row["p95_abs_res"]:>8} {row["dir_accuracy"]:>6}%')

In [ ]:
# R1 bias by planning year (for stability assessment)
print('='*70)
print('R1 BIAS BY PLANNING YEAR')
print('='*70)

if 'planning_year' in r1_data.columns:
    py_col = 'planning_year'
else:
    py_col = None
    for c in r1_data.columns:
        if 'year' in c.lower() or 'py' in c.lower():
            py_col = c
            break

if py_col:
    py_stats = residual_stats(r1_data, [py_col])
    print(f'\n{"PY":<8} {"n":>10} {"Bias":>8} {"Mean|R|":>8} {"p95|R|":>8} {"DirAcc":>7}')
    print('-' * 50)
    for _, row in py_stats.iterrows():
        print(f'{row[py_col]:<8} {row["n"]:>10,} {row["bias"]:>+8} {row["mean_abs_res"]:>8} {row["p95_abs_res"]:>8} {row["dir_accuracy"]:>6}%')
else:
    print('No planning year column found')

In [ ]:
# Find a concrete example path for the worked example in findings_legacy_pricing.md
# Looking for: buy/prevailing path with H ~ 500 in R1 aq1 onpeak
print('='*70)
print('WORKED EXAMPLE: Finding a representative R1 path')
print('='*70)

# Get R1 aq1 onpeak trades
r1_aq1_onpeak = r1_data[(r1_data['period_type'] == 'aq1') & (r1_data['class_type'] == 'onpeak')].copy()
print(f'R1 aq1 onpeak trades: {len(r1_aq1_onpeak)}')

# Find paths with H close to 500
r1_aq1_onpeak['abs_h_diff'] = (r1_aq1_onpeak['mtm_1st_mean'] - 500).abs()
example_candidates = r1_aq1_onpeak.nsmallest(20, 'abs_h_diff')

print(f'\nTop candidates (H closest to 500):')
display_cols = [c for c in ['path', 'source_id', 'sink_id', 'planning_year', 'mtm_1st_mean', 'mcp_mean', 'residual', 'trade_type'] if c in example_candidates.columns]
print(example_candidates[display_cols].head(10).to_string(index=False))

# Pick one specific example
if len(example_candidates) > 0:
    example = example_candidates.iloc[0]
    print(f'\n=== Selected Example ===')
    for col in display_cols:
        print(f'  {col}: {example[col]}')
    print(f'  H (mtm_1st_mean): {example["mtm_1st_mean"]:.2f}')
    print(f'  Actual MCP (mcp_mean): {example["mcp_mean"]:.2f}')
    print(f'  Residual: {example["residual"]:.2f}')

In [ ]:
# Final summary for report writing
print('='*70)
print('FINAL SUMMARY FOR REPORT')
print('='*70)

print(f'\n--- Data ---')
print(f'Total annual trades loaded: {len(annual):,}')
print(f'Total with valid residuals: {len(all_with_baseline):,}')

print(f'\n--- Coverage ---')
for rnd in [1, 2, 3]:
    if rnd == 1:
        total = len(r1_filled)
        valid = r1_filled['mtm_1st_mean'].notna().sum()
    else:
        rnd_all = annual_with_mcp[annual_with_mcp['round'] == rnd]
        total = len(rnd_all)
        valid = rnd_all['mtm_1st_mean'].notna().sum()
    print(f'  R{rnd}: {valid:,}/{total:,} = {100*valid/total:.1f}%')

print(f'\n--- Residual Summary by Round ---')
for rnd in [1, 2, 3]:
    rnd_data = all_with_baseline[all_with_baseline['round'] == rnd]
    print(f'  R{rnd}: n={len(rnd_data):,}, bias={rnd_data["residual"].mean():+.0f}, '
          f'mean|res|={rnd_data["abs_residual"].mean():.0f}, '
          f'p95={rnd_data["abs_residual"].quantile(0.95):.0f}, '
          f'p99={rnd_data["abs_residual"].quantile(0.99):.0f}')

print(f'\n--- Saved Files ---')
for f in ['annual_cleared_all_v2.parquet', 'annual_with_mcp_v2.parquet', 'r1_filled_v2.parquet', 'all_residuals_v2.parquet']:
    path = f'/opt/temp/qianli/annual_research/{f}'
    if os.path.exists(path):
        print(f'  {f}: {os.path.getsize(path)/1e6:.1f} MB')
    else:
        print(f'  {f}: NOT SAVED YET')